# V6 — 10: Figures & Tables (NeurIPS/ICML)

Generates all paper figures and tables from saved `metrics.json` files.

| Figure | Content |
|--------|---------|
| Fig 1  | Main result bar chart: SR ± Wilson CI for all 18 models |
| Fig 2  | ICPE ablation: B4, A1, A2, A3 |
| Fig 3  | SSCP 3-stage: B4 → A4 → A4cc → P1 → P1cc |
| Fig 4  | K-scaling curves: ACT vs ACM3 vs Full @ k∈{50,100,200} |
| Tab 1  | Main results table (LaTeX) |
| Tab 2  | CC ablation table (LaTeX) |

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
import common_v6 as v6

import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

FIG_DIR = v6.OUTPUT_BASE / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ALL_TAGS = ["B1","B2","B3","B4","N1","A1","A2","A3","A4","A4cc","P1","P1cc",
            "K1","K2","K3","K4","K5","K6"]

metrics = v6.load_all_metrics(ALL_TAGS)
print(f"Loaded metrics for: {list(metrics.keys())}")
missing = [t for t in ALL_TAGS if t not in metrics]
if missing:
    print(f"WARNING: missing metrics for {missing}")

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────

def get_sr(tag):
    m = metrics.get(tag, {})
    return m.get("sr", 0.0) or 0.0

def get_ci(tag):
    m = metrics.get(tag, {})
    sr = m.get("sr", 0.0) or 0.0
    lo = m.get("sr_ci_lo", sr) or sr
    hi = m.get("sr_ci_hi", sr) or sr
    return sr - lo, hi - sr  # asymmetric yerr

COLORS = {
    "baseline": "#4878CF",
    "control":  "#6ACC65",
    "ablation": "#D65F5F",
    "full":     "#B47CC7",
    "cc":       "#C4AD66",
    "kscale":   "#77BEDB",
}

TAG_COLOR = {
    "B1":"baseline","B2":"baseline","B3":"baseline","B4":"baseline",
    "N1":"control",
    "A1":"ablation","A2":"ablation","A3":"ablation","A4":"ablation",
    "A4cc":"cc",
    "P1":"full","P1cc":"cc",
    "K1":"kscale","K2":"kscale","K3":"kscale","K4":"kscale",
    "K5":"kscale","K6":"kscale",
}

In [ ]:
# ── Fig 1: Main result bar chart (all 18 models) ──────────────────────────────

MAIN_TAGS = ["B1","B2","B3","B4","N1","A1","A2","A3","A4","A4cc","P1","P1cc"]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(MAIN_TAGS))
srs = [get_sr(t) for t in MAIN_TAGS]
yerrs = np.array([get_ci(t) for t in MAIN_TAGS]).T  # (2, N)
bar_colors = [COLORS[TAG_COLOR[t]] for t in MAIN_TAGS]

bars = ax.bar(x, srs, yerr=yerrs, capsize=4, color=bar_colors, edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(MAIN_TAGS, rotation=30, ha="right", fontsize=10)
ax.set_ylabel("Success Rate", fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title("V6 Main Results — Success Rate (Wilson 95% CI)", fontsize=13)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

legend_patches = [
    mpatches.Patch(color=COLORS[k], label=k.capitalize()) for k in ["baseline","control","ablation","full","cc"]
]
ax.legend(handles=legend_patches, loc="upper left", fontsize=9)

for bar, sr in zip(bars, srs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{sr:.2f}",
            ha="center", va="bottom", fontsize=7)

plt.tight_layout()
fig.savefig(FIG_DIR / "fig1_main_results.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "fig1_main_results.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / 'fig1_main_results.pdf'}")

In [ ]:
# ── Fig 2: ICPE ablation ──────────────────────────────────────────────────────

ICPE_TAGS = ["B4", "A1", "A2", "A3"]
ICPE_LABELS = ["ACM3\n(no ICPE)", "ACM3+ICPE\n[sincos]", "ACM3+ICPE\n[linear]", "ACM3+ICPE\n[full]"]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(ICPE_TAGS))
srs = [get_sr(t) for t in ICPE_TAGS]
yerrs = np.array([get_ci(t) for t in ICPE_TAGS]).T

ax.bar(x, srs, yerr=yerrs, capsize=5, color=[COLORS[TAG_COLOR[t]] for t in ICPE_TAGS],
       edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(ICPE_LABELS, fontsize=10)
ax.set_ylabel("Success Rate"); ax.set_ylim(0, 1.05)
ax.set_title("Fig 2: ICPE Ablation")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig2_icpe_ablation.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 3: SSCP 3-stage comparison ───────────────────────────────────────────

SSCP_TAGS   = ["B4",   "A4",          "A4cc",         "P1",            "P1cc"]
SSCP_LABELS = ["ACM3", "ACM3+SSCP\n[inf]", "ACM3+SSCP\n[CC]",
               "ACM3+ICPE+SSCP\n[inf]", "ACM3+ICPE+SSCP\n[CC]"]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(SSCP_TAGS))
srs = [get_sr(t) for t in SSCP_TAGS]
yerrs = np.array([get_ci(t) for t in SSCP_TAGS]).T

ax.bar(x, srs, yerr=yerrs, capsize=5, color=[COLORS[TAG_COLOR[t]] for t in SSCP_TAGS],
       edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(SSCP_LABELS, fontsize=9)
ax.set_ylabel("Success Rate"); ax.set_ylim(0, 1.05)
ax.set_title("Fig 3: SSCP 3-Stage Comparison (B4 → inference carry → CC trained)")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig3_sscp_stages.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 4: K-scaling curves ───────────────────────────────────────────────────

ks = [50, 100, 200]

act_srs  = [get_sr("K1"), get_sr("B1"), get_sr("K2")]
acm3_srs = [get_sr("K3"), get_sr("B4"), get_sr("K4")]
full_srs = [get_sr("K5"), get_sr("P1"), get_sr("K6")]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, act_srs,  "o-", color=COLORS["baseline"], label="ACT")
ax.plot(ks, acm3_srs, "s-", color=COLORS["ablation"], label="ACM3")
ax.plot(ks, full_srs, "^-", color=COLORS["full"],     label="ACM3+ICPE+SSCP (Full)")
ax.set_xscale("log", base=2); ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.set_xlabel("Chunk size k"); ax.set_ylabel("Success Rate")
ax.set_ylim(0, 1.05); ax.legend(fontsize=9)
ax.set_title("Fig 4: Success Rate vs Chunk Size")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig4_kscale.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Tab 1: Main results table (LaTeX) ────────────────────────────────────────

MAIN_TABLE_TAGS = ["B1","B2","B3","B4","N1","A3","A4","P1"]

lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{V6 Main Results: Success Rate (SR) with 95\% Wilson CI on AlohaTransferCube (n=500).}",
    r"\label{tab:main}",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"Tag & Method & SR & 95\% CI Low & 95\% CI High \\\\",
    r"\midrule",
]

for tag in MAIN_TABLE_TAGS:
    m = metrics.get(tag, {})
    label = v6.MODEL_LABELS.get(tag, tag).replace("_", "\_")
    sr  = f"{m.get('sr', 0):.3f}"       if m.get('sr')       is not None else "─"
    lo  = f"{m.get('sr_ci_lo', 0):.3f}" if m.get('sr_ci_lo') is not None else "─"
    hi  = f"{m.get('sr_ci_hi', 0):.3f}" if m.get('sr_ci_hi') is not None else "─"
    lines.append(f"{tag} & {label} & {sr} & {lo} & {hi} \\\\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
tab1 = "\n".join(lines)
print(tab1)

with open(FIG_DIR / "tab1_main.tex", "w") as f:
    f.write(tab1)

In [ ]:
# ── Tab 2: CC ablation table (LaTeX) ─────────────────────────────────────────

CC_TABLE_TAGS = ["A4", "A4cc", "P1", "P1cc"]
CC_GROUPS = {
    "A4":   "ACM3+SSCP",
    "A4cc": "ACM3+SSCP+CC",
    "P1":   "Full (ICPE+SSCP)",
    "P1cc": "Full (ICPE+SSCP+CC)",
}

lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{SSCP CC Training Ablation. SR on AlohaTransferCube (n=500).}",
    r"\label{tab:cc}",
    r"\begin{tabular}{lcccc}",
    r"\toprule",
    r"Tag & Method & CC & SR & 95\% CI \\\\",
    r"\midrule",
]
for tag in CC_TABLE_TAGS:
    m = metrics.get(tag, {})
    method = CC_GROUPS[tag].replace("_", "\_")
    cc = "Yes" if "cc" in tag else "No"
    sr = f"{m.get('sr', 0):.3f}" if m.get('sr') is not None else "─"
    lo = f"{m.get('sr_ci_lo', 0):.3f}" if m.get('sr_ci_lo') is not None else "─"
    hi = f"{m.get('sr_ci_hi', 0):.3f}" if m.get('sr_ci_hi') is not None else "─"
    ci = f"[{lo}, {hi}]"
    lines.append(f"{tag} & {method} & {cc} & {sr} & {ci} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
tab2 = "\n".join(lines)
print(tab2)

with open(FIG_DIR / "tab2_cc.tex", "w") as f:
    f.write(tab2)

print(f"\nAll figures saved to {FIG_DIR}")